## Q1(a)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 5)

def show_bgr(image, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots()
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
    return ax

In [ ]:
height, width = 440, 660
source_image = np.full((height, width, 3), (244, 243, 239), dtype=np.uint8)
cv2.rectangle(source_image, (48, 40), (612, 390), (40, 110, 170), 5)
cv2.rectangle(source_image, (90, 115), (295, 310), (100, 185, 235), -1)
cv2.circle(source_image, (443, 220), 91, (78, 145, 60), -1)
cv2.line(source_image, (90, 360), (575, 90), (45, 45, 45), 5)
cv2.putText(source_image, 'CV LAB 07', (160, 87), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (15, 15, 15), 3)
cv2.putText(source_image, '2D TRANSFORM', (130, 349), cv2.FONT_HERSHEY_SIMPLEX, 0.82, (10, 10, 10), 2)
show_bgr(source_image, 'Original image')
plt.show()

In [ ]:
translate_matrix = np.float32([[1, 0, 90], [0, 1, 45]])
rotate_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), 30, 1)
scale_matrix = cv2.getRotationMatrix2D((width / 2, height / 2), 0, 0.7)
shear_matrix = np.float32([[1, 0.32, -55], [0.14, 1, -35]])

transforms_2d = {
    'Translation': cv2.warpAffine(source_image, translate_matrix, (width, height)),
    'Rotation': cv2.warpAffine(source_image, rotate_matrix, (width, height)),
    'Scaling': cv2.warpAffine(source_image, scale_matrix, (width, height)),
    'Shearing': cv2.warpAffine(source_image, shear_matrix, (width, height)),
}

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 4))
show_bgr(source_image, 'Original', axes[0])
for ax, (name, image) in zip(axes[1:], transforms_2d.items()):
    show_bgr(image, name, ax)
plt.tight_layout()
plt.show()

## Q1(b)

In [ ]:
cube = np.array([
    [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
    [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]
], dtype=float)
edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]

def rotation_3d(axis, degrees):
    t = np.deg2rad(degrees)
    c, s = np.cos(t), np.sin(t)
    if axis == 'x':
        return np.array([[1,0,0],[0,c,-s],[0,s,c]])
    if axis == 'y':
        return np.array([[c,0,s],[0,1,0],[-s,0,c]])
    return np.array([[c,-s,0],[s,c,0],[0,0,1]])

In [ ]:
cube_versions = {
    'Original': cube,
    'Rotate X': cube @ rotation_3d('x', 40).T,
    'Rotate Y': cube @ rotation_3d('y', 45).T,
    'Rotate Z': cube @ rotation_3d('z', 45).T,
    'Translate + scale': cube * 0.7 + np.array([0.9, 0.5, 0.6])
}

In [ ]:
fig = plt.figure(figsize=(17, 4))
for i, (name, points) in enumerate(cube_versions.items(), 1):
    ax = fig.add_subplot(1, 5, i, projection='3d')
    for a, b in edges:
        ax.plot(*points[[a,b]].T, color='royalblue', linewidth=2)
    ax.scatter(*points.T, s=16, color='tomato')
    ax.set(xlim=(-2,2), ylim=(-2,2), zlim=(-2,2), title=name,
           xlabel='X', ylabel='Y', zlabel='Z')
    ax.set_box_aspect((1,1,1))
plt.tight_layout()
plt.show()

## Q1(c)

In [ ]:
point_2d = np.array([120, 80], dtype=float)
point_h = np.append(point_2d, 1.0)
translation_h = np.array([[1, 0, 65], [0, 1, 40], [0, 0, 1]], dtype=float)
transformed_h = translation_h @ point_h
transformed_2d = transformed_h[:2] / transformed_h[2]
print('Original point:', point_2d)
print('Homogeneous point:', point_h)
print('After translation:', transformed_2d)
print('The extra coordinate allows translation and other transformations through matrix multiplication.')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(*point_2d, s=90, label='Original (120, 80)')
ax.scatter(*transformed_2d, s=90, label='Transformed (185, 120)')
ax.annotate('', xy=transformed_2d, xytext=point_2d,
            arrowprops={'arrowstyle':'->', 'linewidth':2})
ax.set(xlim=(0,250), ylim=(0,180), xlabel='X', ylabel='Y')
ax.grid(True)
ax.legend()
plt.show()

## Q1(d)

In [ ]:
affine_matrix = np.float32([[1.05, 0.25, 15], [-0.12, 0.90, 70]])
affine_image = cv2.warpAffine(source_image, affine_matrix, (width, height))
print('Affine matrix:\n', affine_matrix)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
show_bgr(source_image, 'Before affine transformation', axes[0])
show_bgr(affine_image, 'After affine transformation', axes[1])
plt.tight_layout()
plt.show()
print('Straight lines remain straight; parallel lines remain parallel under an affine transformation.')

## Q1(e)

In [ ]:
flat_corners = np.float32([[0, 0], [width-1, 0], [width-1, height-1], [0, height-1]])
tilted_corners = np.float32([[175, 170], [795, 80], [900, 650], [105, 735]])
scene_size = (1020, 830)
homography_tilt = cv2.getPerspectiveTransform(flat_corners, tilted_corners)
tilted_image = cv2.warpPerspective(source_image, homography_tilt, scene_size,
                                  borderMode=cv2.BORDER_CONSTANT,
                                  borderValue=(230, 230, 230))

In [ ]:
homography_correct = cv2.getPerspectiveTransform(tilted_corners, flat_corners)
perspective_corrected = cv2.warpPerspective(tilted_image, homography_correct, (width, height))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
show_bgr(source_image, 'Flat image', axes[0])
show_bgr(tilted_image, 'Tilted perspective', axes[1])
show_bgr(perspective_corrected, 'Corrected homography', axes[2])
plt.tight_layout()
plt.show()

## Q2

In [ ]:
def apply_perspective_transform(image, source_points, destination_points, output_size):
    src = np.asarray(source_points, dtype=np.float32)
    dst = np.asarray(destination_points, dtype=np.float32)
    matrix = cv2.getPerspectiveTransform(src, dst)
    transformed = cv2.warpPerspective(image, matrix, output_size)
    return transformed, matrix

In [ ]:
rectangle_points = np.float32([[0,0], [width-1,0], [width-1,height-1], [0,height-1]])
unwrapped, inverse_h = apply_perspective_transform(
    tilted_image, tilted_corners, rectangle_points, (width, height)
)
print('Inverse homography:\n', inverse_h)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
preview = tilted_image.copy()
cv2.polylines(preview, [tilted_corners.astype(np.int32)], True, (0, 0, 255), 4)
show_bgr(preview, 'Source: tilted quadrilateral', axes[0])
show_bgr(unwrapped, 'Destination: upright rectangle', axes[1])
plt.tight_layout()
plt.show()

In [ ]:
PHOTO_PATH = None
REAL_PHOTO_CORNERS = None
if PHOTO_PATH and REAL_PHOTO_CORNERS is not None:
    real_photo = cv2.imread(str(PHOTO_PATH))
    if real_photo is None:
        raise FileNotFoundError(PHOTO_PATH)
    real_unwrapped, real_H = apply_perspective_transform(
        real_photo, REAL_PHOTO_CORNERS, rectangle_points, (width, height)
    )
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    show_bgr(real_photo, 'Real tilted photo', axes[0])
    show_bgr(real_unwrapped, 'Rectified photo', axes[1])
    plt.show()

## Q3

In [ ]:
calibration_dir = Path('lab07_checkerboard_photos')
calibration_dir.mkdir(exist_ok=True)
calibration_file = Path('lab07_camera_calibration.npz')
CHECKERBOARD = (9, 6)
SQUARE_SIZE_MM = 25.0
CAMERA_INDEX = 0

In [ ]:
square_px, margin = 76, 40
board_width = (CHECKERBOARD[0] + 1) * square_px
board_height = (CHECKERBOARD[1] + 1) * square_px
board = np.full((board_height + 2 * margin, board_width + 2 * margin, 3), 255, np.uint8)
for y in range(CHECKERBOARD[1] + 1):
    for x in range(CHECKERBOARD[0] + 1):
        if (x + y) % 2 == 0:
            x0, y0 = margin + x * square_px, margin + y * square_px
            cv2.rectangle(board, (x0, y0), (x0 + square_px, y0 + square_px), (0,0,0), -1)
cv2.imwrite('lab07_checkerboard_pattern.png', board)
show_bgr(board, '9 x 6 inner-corner calibration board')
plt.show()

In [ ]:
CAPTURE_CALIBRATION_PHOTOS = True
TARGET_CALIBRATION_PHOTOS = 15
if CAPTURE_CALIBRATION_PHOTOS and not list(calibration_dir.glob('*.jpg')):
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print('Webcam unavailable. Add checkerboard images to', calibration_dir.resolve())
    else:
        saved = 0
        print("Move the checkerboard through varied angles. Press 's' to save, 'q' to finish.")
        try:
            while saved < TARGET_CALIBRATION_PHOTOS:
                ok, frame = cap.read()
                if not ok:
                    break
                gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                found, corners = cv2.findChessboardCorners(gray, CHECKERBOARD)
                preview = frame.copy()
                if found:
                    cv2.drawChessboardCorners(preview, CHECKERBOARD, corners, found)
                cv2.putText(preview, f'Saved {saved}/{TARGET_CALIBRATION_PHOTOS}  s=save q=quit',
                            (12, 30), cv2.FONT_HERSHEY_SIMPLEX, .65, (0, 255, 0), 2)
                cv2.imshow('Calibration capture', preview)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('s') and found:
                    cv2.imwrite(str(calibration_dir / f'checkerboard_{saved:03d}.jpg'), frame)
                    saved += 1
                elif key in (ord('q'), 27):
                    break
        finally:
            cap.release()
            cv2.destroyAllWindows()
        print('Saved calibration images:', saved)
else:
    print('Existing images:', len(list(calibration_dir.glob('*.jpg'))))

In [ ]:
square_grid = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
square_grid[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
square_grid *= SQUARE_SIZE_MM

object_points, image_points = [], []
image_size = None
for filename in sorted(calibration_dir.glob('*.jpg')):
    image = cv2.imread(str(filename))
    if image is None:
        continue
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    current_size = gray.shape[::-1]
    if image_size is not None and image_size != current_size:
        continue
    found, corners = cv2.findChessboardCorners(gray, CHECKERBOARD)
    if found:
        refined = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1),
                                   (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, .001))
        object_points.append(square_grid.copy())
        image_points.append(refined)
        image_size = current_size
print('Usable checkerboard views:', len(object_points))

In [ ]:
if len(object_points) >= 5:
    reprojection_rms, camera_matrix, distortion, rvecs, tvecs = cv2.calibrateCamera(
        object_points, image_points, image_size, None, None
    )
    np.savez(calibration_file, camera_matrix=camera_matrix, distortion=distortion,
             image_size=image_size, rms=reprojection_rms)
    print('Camera matrix:\n', camera_matrix)
    print('Distortion coefficients:', distortion.ravel())
    print('Reprojection RMS (pixels):', reprojection_rms)
    print('Saved:', calibration_file.resolve())
else:
    print('Need at least 5 usable checkerboard photos to calibrate (10-20 recommended).')

In [ ]:
if calibration_file.exists():
    data = np.load(calibration_file)
    camera_matrix, distortion = data['camera_matrix'], data['distortion']
    photo_list = sorted(calibration_dir.glob('*.jpg'))
    if photo_list:
        distorted = cv2.imread(str(photo_list[0]))
        undistorted = cv2.undistort(distorted, camera_matrix, distortion)
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        show_bgr(distorted, 'Before undistortion', axes[0])
        show_bgr(undistorted, 'After undistortion', axes[1])
        plt.tight_layout()
        plt.show()
else:
    print('Calibrate first to produce', calibration_file)

In [ ]:
if calibration_file.exists():
    data = np.load(calibration_file)
    K, dist = data['camera_matrix'], data['distortion']
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print('Webcam unavailable.')
    else:
        print('Press q or Esc to stop the live undistorted camera.')
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                result = cv2.undistort(frame, K, dist)
                display = np.hstack([frame, result])
                cv2.putText(display, 'Original', (15, 30), cv2.FONT_HERSHEY_SIMPLEX, .8, (0,255,0), 2)
                cv2.putText(display, 'Undistorted', (frame.shape[1]+15, 30),
                            cv2.FONT_HERSHEY_SIMPLEX, .8, (0,255,0), 2)
                cv2.imshow('Q3: Live camera undistortion', display)
                if (cv2.waitKey(1) & 0xFF) in (ord('q'), 27):
                    break
        finally:
            cap.release()
            cv2.destroyAllWindows()
else:
    print('Calibration file missing; complete the earlier calibration cells first.')

## Q4

In [ ]:
import shutil
import subprocess

colmap_project = Path('lab07_colmap_project').resolve()
colmap_images = colmap_project / 'images'
colmap_sparse = colmap_project / 'sparse'
colmap_dense = colmap_project / 'dense'
colmap_database = colmap_project / 'database.db'
for folder in [colmap_images, colmap_sparse, colmap_dense]:
    folder.mkdir(parents=True, exist_ok=True)
COLMAP_EXECUTABLE = shutil.which('colmap') or shutil.which('COLMAP.bat')
print('COLMAP executable:', COLMAP_EXECUTABLE or 'Not found on PATH')
print('Image folder:', colmap_images)

In [ ]:
CAPTURE_COLMAP_IMAGES = True
MIN_COLMAP_IMAGES = 15
existing = sorted(p for p in colmap_images.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})
if CAPTURE_COLMAP_IMAGES and len(existing) < MIN_COLMAP_IMAGES:
    camera = cv2.VideoCapture(CAMERA_INDEX)
    if not camera.isOpened():
        print('Camera unavailable. Add 15+ overlapping views of a stationary textured scene to', colmap_images)
    else:
        count = len(existing)
        print("Move the CAMERA around a stationary scene with overlap and parallax. Press 's' to save, 'q' to finish.")
        try:
            while True:
                ok, frame = camera.read()
                if not ok:
                    break
                cv2.putText(frame, f'Photos: {count}   s=save   q=finish', (12, 28),
                            cv2.FONT_HERSHEY_SIMPLEX, .7, (0,255,0), 2)
                cv2.imshow('COLMAP image capture', frame)
                key = cv2.waitKey(1) & 0xFF
                if key == ord('s'):
                    filename = colmap_images / f'view_{count:03d}.jpg'
                    cv2.imwrite(str(filename), frame)
                    count += 1
                elif key in (ord('q'), 27):
                    break
        finally:
            camera.release()
            cv2.destroyAllWindows()
image_files = sorted(p for p in colmap_images.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png'})
print('COLMAP images:', len(image_files))

In [ ]:
if image_files:
    examples = image_files[:min(6, len(image_files))]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax in axes.flat:
        ax.axis('off')
    for ax, photo in zip(axes.flat, examples):
        show_bgr(cv2.imread(str(photo)), photo.name, ax)
    plt.tight_layout()
    plt.show()
else:
    print('No images in', colmap_images)

In [ ]:
colmap_ready = bool(COLMAP_EXECUTABLE) and len(image_files) >= MIN_COLMAP_IMAGES

def run_colmap(*args):
    command = [str(COLMAP_EXECUTABLE), *map(str, args)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)

if not COLMAP_EXECUTABLE:
    print('Install COLMAP and add colmap/COLMAP.bat to PATH, or set COLMAP_EXECUTABLE manually.')
elif len(image_files) < MIN_COLMAP_IMAGES:
    print('Add at least', MIN_COLMAP_IMAGES, 'overlapping photographs before reconstruction.')

In [ ]:
if colmap_ready:
    run_colmap('feature_extractor', '--database_path', colmap_database,
               '--image_path', colmap_images, '--ImageReader.single_camera', '1',
               '--SiftExtraction.use_gpu', '0')

In [ ]:
if colmap_ready and colmap_database.exists():
    run_colmap('exhaustive_matcher', '--database_path', colmap_database,
               '--SiftMatching.use_gpu', '0')

In [ ]:
if colmap_ready and colmap_database.exists():
    run_colmap('mapper', '--database_path', colmap_database,
               '--image_path', colmap_images, '--output_path', colmap_sparse)

In [ ]:
sparse_models = sorted(
    p for p in colmap_sparse.iterdir()
    if p.is_dir() and (p / 'images.bin').exists()
)
sparse_model = sparse_models[0] if sparse_models else None
print('Sparse model:', sparse_model or 'Not created; check the image overlap and COLMAP output.')

In [ ]:
sparse_ply = colmap_project / 'sparse_point_cloud.ply'
if colmap_ready and sparse_model is not None:
    run_colmap('model_converter', '--input_path', sparse_model,
               '--output_path', sparse_ply, '--output_type', 'PLY')
    print('Sparse point cloud:', sparse_ply)

In [ ]:
model_txt = colmap_project / 'model_txt'
if colmap_ready and sparse_model is not None:
    model_txt.mkdir(exist_ok=True)
    run_colmap('model_converter', '--input_path', sparse_model,
               '--output_path', model_txt, '--output_type', 'TXT')
    poses_file = model_txt / 'images.txt'
    for line in poses_file.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if len(parts) >= 10 and parts[-1].lower().endswith(('.jpg','.jpeg','.png')):
            print('Image:', parts[-1], 'Quaternion:', parts[1:5], 'Translation:', parts[5:8])

In [ ]:
points_txt = model_txt / 'points3D.txt'
if points_txt.exists():
    point_rows = [line.split() for line in points_txt.read_text(encoding='utf-8').splitlines()
                  if line.strip() and not line.startswith('#')]
    coordinates = np.array([[float(row[1]), float(row[2]), float(row[3])]
                            for row in point_rows if len(row) >= 7])
    colors = np.array([[int(row[4]), int(row[5]), int(row[6])]
                       for row in point_rows if len(row) >= 7]) / 255.0
    if len(coordinates):
        indices = np.linspace(0, len(coordinates)-1, min(15000, len(coordinates))).astype(int)
        fig = plt.figure(figsize=(9, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.scatter(*coordinates[indices].T, c=colors[indices], s=1)
        ax.set_title(f'Sparse reconstruction: {len(coordinates):,} points')
        plt.show()
        print('Reconstructed 3D points:', len(coordinates))
else:
    print('Point cloud unavailable until sparse mapping succeeds.')

In [ ]:
sparse_mesh = colmap_project / 'sparse_mesh.ply'
if colmap_ready and sparse_model is not None:
    try:
        run_colmap('delaunay_mesher', '--input_path', sparse_model,
                   '--input_type', 'sparse', '--output_path', sparse_mesh)
    except subprocess.CalledProcessError:
        print('COLMAP sparse meshing is unavailable in this build. Use the next cell for a coarse fallback.')

In [ ]:
if not sparse_mesh.exists() and 'coordinates' in globals() and len(coordinates) >= 4:
    from scipy.spatial import ConvexHull
    coords = coordinates.astype(np.float64)
    if len(coords) > 15000:
        coords = coords[np.linspace(0, len(coords)-1, 15000).astype(int)]
    try:
        hull = ConvexHull(coords)
        faces = hull.simplices
        with sparse_mesh.open('w', encoding='utf-8') as file:
            file.write('ply\nformat ascii 1.0\n')
            file.write(f'element vertex {len(coords)}\nproperty float x\nproperty float y\nproperty float z\n')
            file.write(f'element face {len(faces)}\nproperty list uchar int vertex_indices\nend_header\n')
            for p in coords:
                file.write(f'{p[0]} {p[1]} {p[2]}\n')
            for face in faces:
                file.write(f'3 {face[0]} {face[1]} {face[2]}\n')
        print('Coarse convex-hull mesh (not a detailed reconstructed surface):', sparse_mesh)
    except Exception as exc:
        print('Coarse mesh could not be generated:', exc)

In [ ]:
RUN_DENSE_RECONSTRUCTION = False
if RUN_DENSE_RECONSTRUCTION and colmap_ready and sparse_model is not None:
    run_colmap('image_undistorter', '--image_path', colmap_images,
               '--input_path', sparse_model, '--output_path', colmap_dense,
               '--output_type', 'COLMAP', '--max_image_size', '1600')

In [ ]:
if RUN_DENSE_RECONSTRUCTION and (colmap_dense / 'sparse').exists():
    run_colmap('patch_match_stereo', '--workspace_path', colmap_dense,
               '--workspace_format', 'COLMAP', '--PatchMatchStereo.geom_consistency', 'true')

In [ ]:
dense_ply = colmap_dense / 'fused.ply'
if RUN_DENSE_RECONSTRUCTION and (colmap_dense / 'stereo').exists():
    run_colmap('stereo_fusion', '--workspace_path', colmap_dense,
               '--workspace_format', 'COLMAP', '--input_type', 'geometric',
               '--output_path', dense_ply)

In [ ]:
dense_mesh = colmap_dense / 'meshed-poisson.ply'
if RUN_DENSE_RECONSTRUCTION and dense_ply.exists():
    run_colmap('poisson_mesher', '--input_path', dense_ply, '--output_path', dense_mesh)

In [ ]:
for label, file in {
    'Sparse point cloud': sparse_ply,
    'Sparse mesh': sparse_mesh,
    'Dense point cloud': dense_ply,
    'Dense mesh': dense_mesh,
}.items():
    print(f'{label}:', str(file) if file.exists() else 'Not produced')